In [1]:
# ============================================================
#  DIRICHLET CONVOLUTION — ONE-CELL VALIDATION (FIXED)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ──────────────────────────────────────────────
# SECTION 1: The Dirichlet Convolution Layer
# ──────────────────────────────────────────────
class DirichletConv1D(nn.Module):
    """
    Y_n = sum_{k | n} W_k @ X_{n/k}
    O(N log N) edges instead of O(N^2)
    """
    def __init__(self, N, in_dim, out_dim):
        super().__init__()
        self.N = N
        self.in_dim = in_dim
        self.out_dim = out_dim
        
        self.W = nn.Parameter(torch.randn(N, in_dim, out_dim) * 0.02)
        
        out_idx, in_idx, w_idx = self._build_divisor_graph(N)
        self.register_buffer('out_idx', out_idx)
        self.register_buffer('in_idx', in_idx)
        self.register_buffer('w_idx', w_idx)
        self.num_edges = len(out_idx)
    
    def _build_divisor_graph(self, N):
        out_idx, in_idx, w_idx = [], [], []
        for n in range(1, N + 1):
            for k in range(1, n + 1):
                if n % k == 0:
                    out_idx.append(n - 1)
                    in_idx.append((n // k) - 1)
                    w_idx.append(k - 1)
        return (torch.tensor(out_idx, dtype=torch.long),
                torch.tensor(in_idx, dtype=torch.long),
                torch.tensor(w_idx, dtype=torch.long))
    
    def forward(self, X):
        # X: [B, N, in_dim]
        B, N, D_in = X.shape
        
        # ✅ FIX: Gather directly from batched tensor (no flattening)
        X_src = X[:, self.in_idx, :]          # [B, num_edges, D_in]
        W_active = self.W[self.w_idx]          # [num_edges, D_in, D_out]
        
        # Batched multiply along edges
        Y_edges = torch.einsum('bed,edo->beo', X_src, W_active)
        
        # Scatter-add to output positions
        Y = torch.zeros(B, N, self.out_dim, device=X.device, dtype=X.dtype)
        Y.index_add_(1, self.out_idx, Y_edges)
        return Y


# ──────────────────────────────────────────────
# SECTION 2: Tiny Model
# ──────────────────────────────────────────────
class DirichletDenoiser(nn.Module):
    def __init__(self, N, D):
        super().__init__()
        self.conv1 = DirichletConv1D(N, D, D)
        self.conv2 = DirichletConv1D(N, D, D)
        self.act = nn.GELU()
        
        # Möbius residual init: W_1 ≈ Identity
        with torch.no_grad():
            self.conv1.W[0] = torch.eye(D) * 0.9
            self.conv2.W[0] = torch.eye(D) * 0.9
    
    def forward(self, x):
        h = self.act(self.conv1(x))
        return self.conv2(h)


# ──────────────────────────────────────────────
# SECTION 3: Data
# ──────────────────────────────────────────────
N = 128; D = 32; BATCH = 32; STEPS = 300

def make_signal(batch_size, N):
    t = torch.linspace(0, 4 * np.pi, N)
    clean = (torch.sin(1.0 * t) + 0.5 * torch.sin(3.7 * t) + 0.3 * torch.sin(7.3 * t))
    clean = clean.unsqueeze(0).unsqueeze(-1).expand(batch_size, N, D)
    noise = torch.randn_like(clean) * 0.3
    return clean, clean + noise

print(f"N={N}, D={D}")


# ──────────────────────────────────────────────
# SECTION 4: Setup
# ──────────────────────────────────────────────
model = DirichletDenoiser(N, D).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

total_params = sum(p.numel() for p in model.parameters())
dense_edges = N * N
dirichlet_edges = model.conv1.num_edges

print(f"\n{'='*50}")
print(f"  ARCHITECTURE STATS")
print(f"{'='*50}")
print(f"  Total parameters:    {total_params:,}")
print(f"  Dense edges (N²):    {dense_edges:,}")
print(f"  Dirichlet edges:     {dirichlet_edges:,}")
print(f"  Compression:         {dense_edges/dirichlet_edges:.1f}x fewer edges")
print(f"{'='*50}\n")


# ──────────────────────────────────────────────
# SECTION 5: Train
# ──────────────────────────────────────────────
losses = []
start_time = time.time()

print("Training...")
for step in range(STEPS):
    clean, noisy = make_signal(BATCH, N)
    clean, noisy = clean.to(device), noisy.to(device)
    
    pred = model(noisy)
    loss = loss_fn(pred, clean)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if (step + 1) % 50 == 0:
        print(f"  Step {step+1:3d}/{STEPS} | Loss: {loss.item():.6f}")

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s")


# ──────────────────────────────────────────────
# SECTION 6: Visualize
# ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(losses, color='#2196F3', linewidth=1.5)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('MSE')
axes[0].grid(True, alpha=0.3)

model.eval()
with torch.no_grad():
    clean_t, noisy_t = make_signal(1, N)
    clean_t, noisy_t = clean_t.to(device), noisy_t.to(device)
    pred_t = model(noisy_t)
    
    t = np.linspace(0, 4*np.pi, N)
    axes[1].plot(t, noisy_t[0,:,0].cpu(), alpha=0.3, color='gray', label='Noisy')
    axes[1].plot(t, clean_t[0,:,0].cpu(), color='#4CAF50', lw=2, label='Clean')
    axes[1].plot(t, pred_t[0,:,0].cpu(), color='#FF5722', lw=2, ls='--', label='Dirichlet')
    axes[1].set_title('Reconstruction', fontweight='bold')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

axes[2].scatter(model.conv1.in_idx.cpu(), model.conv1.out_idx.cpu(),
                s=0.5, alpha=0.3, color='#9C27B0')
axes[2].set_title(f'Divisor Graph ({dirichlet_edges} edges)', fontweight='bold')
axes[2].set_xlabel('Input (n/k)'); axes[2].set_ylabel('Output (n)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dirichlet_validation.png', dpi=150)
plt.show()

# ──────────────────────────────────────────────
# VERDICT
# ──────────────────────────────────────────────
improvement = (1 - losses[-1] / losses[0]) * 100
print(f"\n{'='*50}")
print(f"  VERDICT")
print(f"{'='*50}")
print(f"  Initial loss:  {losses[0]:.6f}")
print(f"  Final loss:    {losses[-1]:.6f}")
print(f"  Improvement:   {improvement:.1f}%")

if improvement > 50:
    print(f"\n  ✅ SUCCESS: Dirichlet Convolution LEARNS!")
else:
    print(f"\n  ⚠️  WEAK: Needs tuning.")
print(f"{'='*50}")

KeyboardInterrupt: 

In [ ]:
# ============================================================
#  EXPERIMENT 2: THE LOCAL-CONTEXT STRESS TEST
#  Pure Dirichlet vs Pure Conv1D vs Hybrid (Parameter-Matched)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ──────────────────────────────────────────────
# 1. THE PRIMITIVES
# ──────────────────────────────────────────────
class DirichletConv1D(nn.Module):
    def __init__(self, N, in_dim, out_dim):
        super().__init__()
        self.N = N
        self.W = nn.Parameter(torch.randn(N, in_dim, out_dim) * 0.02)
        out_idx, in_idx, w_idx = self._build_graph(N)
        self.register_buffer('out_idx', out_idx)
        self.register_buffer('in_idx', in_idx)
        self.register_buffer('w_idx', w_idx)
        
    def _build_graph(self, N):
        out_idx, in_idx, w_idx = [], [], []
        for n in range(1, N + 1):
            for k in range(1, n + 1):
                if n % k == 0:
                    out_idx.append(n - 1)
                    in_idx.append((n // k) - 1)
                    w_idx.append(k - 1)
        return (torch.tensor(out_idx, dtype=torch.long),
                torch.tensor(in_idx, dtype=torch.long),
                torch.tensor(w_idx, dtype=torch.long))
                
    def forward(self, X):
        B, N, D_in = X.shape
        X_src = X[:, self.in_idx, :]          
        W_active = self.W[self.w_idx]          
        Y_edges = torch.einsum('bed,edo->beo', X_src, W_active)
        Y = torch.zeros(B, N, self.W.shape[-1], device=X.device, dtype=X.dtype)
        Y.index_add_(1, self.out_idx, Y_edges)
        return Y

class LocalConvBlock(nn.Module):
    def __init__(self, D, D_hidden):
        super().__init__()
        # kernel_size=3 as strictly mandated by The Theorist
        self.c1 = nn.Conv1d(D, D_hidden, kernel_size=3, padding=1)
        self.c2 = nn.Conv1d(D_hidden, D, kernel_size=3, padding=1)
        self.act = nn.GELU()
        
    def forward(self, X):
        # X: [B, N, D] -> Conv1d expects [B, D, N]
        h = self.act(self.c1(X.transpose(1, 2)))
        return self.c2(h).transpose(1, 2)

# ──────────────────────────────────────────────
# 2. THE THREE COMPETING MODELS
# ──────────────────────────────────────────────
N = 128
D = 32
# MATHEMATICAL PARAMETER MATCHING:
# Dirichlet params = N * D * D. 
# Conv params = 2 * 3 * D * D_hidden = 6 * D * D_hidden.
# Equating them: D_hidden = (N * D) / 6
D_hidden = (N * D) // 6  

class PureDirichlet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DirichletConv1D(N, D, D)
        self.d2 = DirichletConv1D(N, D, D)
        self.act = nn.GELU()
    def forward(self, x): return self.d2(self.act(self.d1(x)))

class PureConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = LocalConvBlock(D, D_hidden)
        self.c2 = LocalConvBlock(D, D_hidden)
    def forward(self, x): return self.c2(torch.relu(self.c1(x)))

class Hybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DirichletConv1D(N, D, D)
        self.c1 = LocalConvBlock(D, D_hidden) # Fuses O(N log N) + O(N*K)
    def forward(self, x): return self.c1(torch.relu(self.d1(x)))

# ──────────────────────────────────────────────
# 3. THE "SPIKY" DATA (The Blindness Test)
# ──────────────────────────────────────────────
def make_spiky_signal(B, N):
    t = torch.linspace(0, 4*np.pi, N)
    base = torch.sin(1.0 * t) # Global structure
    spikes = torch.zeros(N)
    # Inject sharp local bottlenecks (mimicking discrete tokens/logic)
    spike_idx = torch.randint(5, N-5, (N//8,)) 
    spikes[spike_idx] = torch.randn(N//8) * 2.5 
    clean = base + spikes
    clean = clean.unsqueeze(0).unsqueeze(-1).expand(B, N, D)
    return clean, clean + torch.randn_like(clean) * 0.3

# ──────────────────────────────────────────────
# 4. TRAINING LOOP
# ──────────────────────────────────────────────
models = {
    'Pure Dirichlet': PureDirichlet().to(device),
    'Pure Conv1D': PureConv().to(device),
    'Hybrid': Hybrid().to(device)
}

print(f"\n{'='*60}")
print(f"  CAPACITY CONTROL (Parameter Matching)")
print(f"{'='*60}")
for name, model in models.items():
    params = sum(p.numel() for p in model.parameters())
    print(f"  {name:15} | Params: {params:,}")
print(f"{'='*60}\n")

history = {name: [] for name in models}
optimizers = {name: torch.optim.Adam(m.parameters(), lr=2e-3) for name, m in models.items()}
loss_fn = nn.MSELoss()
STEPS = 400

print("Training on Spiky Signal...")
start = time.time()
for step in range(STEPS):
    clean, noisy = make_signal(32, N)
    clean, noisy = clean.to(device), noisy.to(device)
    
    for name, model in models.items():
        pred = model(noisy)
        loss = loss_fn(pred, clean)
        optimizers[name].zero_grad()
        loss.backward()
        optimizers[name].step()
        history[name].append(loss.item())
        
    if (step+1) % 100 == 0:
        print(f"Step {step+1:3d} | " + " | ".join([f"{n}: {history[n][-1]:.4f}" for n in models]))
print(f"Done in {time.time()-start:.1f}s")

# ──────────────────────────────────────────────
# 5. THE SMOKING GUN VISUALIZATION
# ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Loss Curves
for name, losses in history.items():
    axes[0,0].plot(losses, label=name, linewidth=2)
axes[0,0].set_title('Training Loss (Spiky Signal)', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Plot 2: Global Reconstruction
model_eval = {name: m.eval() for name, m in models.items()}
clean_t, noisy_t = make_spiky_signal(1, N)
clean_t, noisy_t = clean_t.to(device), noisy_t.to(device)

t = np.arange(N)
axes[0,1].plot(t, noisy_t[0,:,0].cpu(), alpha=0.2, color='gray', label='Noisy')
axes[0,1].plot(t, clean_t[0,:,0].cpu(), color='black', lw=2, label='True Signal')
for name, model in models.items():
    with torch.no_grad():
        pred = model(noisy_t)
    axes[0,1].plot(t, pred[0,:,0].cpu(), lw=1.5, ls='--', label=name)
axes[0,1].set_title('Global Reconstruction', fontweight='bold')
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

# Plot 3: ZOOM IN ON A SPIKE (The Blindness Test)
# Find the first major spike in the test data
spike_loc = torch.argmax(torch.abs(clean_t[0, :, 0] - torch.sin(1.0 * torch.linspace(0, 4*np.pi, N).to(device)))).item()
zoom_start = max(0, spike_loc - 10)
zoom_end = min(N, spike_loc + 10)

axes[1,0].plot(t[zoom_start:zoom_end], noisy_t[0,zoom_start:zoom_end,0].cpu(), alpha=0.3, color='gray')
axes[1,0].plot(t[zoom_start:zoom_end], clean_t[0,zoom_start:zoom_end,0].cpu(), color='black', lw=3, label='True Spike')
for name, model in models.items():
    with torch.no_grad():
        pred = model(noisy_t)
    axes[1,0].plot(t[zoom_start:zoom_end], pred[0,zoom_start:zoom_end,0].cpu(), lw=2, ls='--', label=name)
axes[1,0].set_title(f'ZOOM IN: Local Spike Blindness Test (Index {zoom_start}-{zoom_end})', fontweight='bold', color='red')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True, alpha=0.3)

# Plot 4: Final MSE Bar Chart
final_mse = {name: history[name][-1] for name in models}
axes[1,1].bar(final_mse.keys(), final_mse.values(), color=['#9C27B0', '#4CAF50', '#FF5722'])
axes[1,1].set_title('Final MSE (Lower is Better)', fontweight='bold')
axes[1,1].set_ylabel('Mean Squared Error')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('experiment2_stress_test.png', dpi=150)
plt.show()

# VERDICT
print("\n" + "="*60)
print("  EMPIRICAL VERDICT")
print("="*60)
best_model = min(final_mse, key=final_mse.get)
print(f"  Winner: {best_model} (MSE: {final_mse[best_model]:.5f})")
if best_model == 'Hybrid':
    print("  ✅ HYPOTHESIS CONFIRMED: The Hybrid successfully merges")
    print("     global O(N log N) routing with local O(N*K) spike detection.")
print("="*60)

In [ ]:
# ============================================================
#  EXPERIMENT 3: MASKED INPAINTING STRESS TEST
#  Definitive Proof of Hybrid Dominance via Gap-Filling
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ──────────────────────────────────────────────
# 1. THE PRIMITIVES (Identical to Exp 2)
# ──────────────────────────────────────────────
class DirichletConv1D(nn.Module):
    def __init__(self, N, D):
        super().__init__()
        self.N = N
        self.W = nn.Parameter(torch.randn(N, D, D) * 0.02)
        out_idx, in_idx, w_idx = self._build_graph(N)
        self.register_buffer('out_idx', out_idx)
        self.register_buffer('in_idx', in_idx)
        self.register_buffer('w_idx', w_idx)
        
    def _build_graph(self, N):
        out_idx, in_idx, w_idx = [], [], []
        for n in range(1, N + 1):
            for k in range(1, n + 1):
                if n % k == 0:
                    out_idx.append(n - 1)
                    in_idx.append((n // k) - 1)
                    w_idx.append(k - 1)
        return (torch.tensor(out_idx, dtype=torch.long),
                torch.tensor(in_idx, dtype=torch.long),
                torch.tensor(w_idx, dtype=torch.long))
                
    def forward(self, X):
        B, N, D = X.shape
        X_src = X[:, self.in_idx, :]          
        W_active = self.W[self.w_idx]          
        Y_edges = torch.einsum('bed,edo->beo', X_src, W_active)
        Y = torch.zeros(B, N, D, device=X.device, dtype=X.dtype)
        Y.index_add_(1, self.out_idx, Y_edges)
        return Y

class LocalConvBlock(nn.Module):
    def __init__(self, D, D_hidden):
        super().__init__()
        self.c1 = nn.Conv1d(D, D_hidden, kernel_size=3, padding=1)
        self.c2 = nn.Conv1d(D_hidden, D, kernel_size=3, padding=1)
        self.act = nn.GELU()
        
    def forward(self, X):
        h = self.act(self.c1(X.transpose(1, 2)))
        return self.c2(h).transpose(1, 2)

# ──────────────────────────────────────────────
# 2. PARAMETER-MATCHED MODELS
# ──────────────────────────────────────────────
N = 128; D = 32
D_hidden = (N * D) // 6 

class PureDirichlet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DirichletConv1D(N, D); self.d2 = DirichletConv1D(N, D)
    def forward(self, x): return self.d2(torch.relu(self.d1(x)))

class PureConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = LocalConvBlock(D, D_hidden); self.c2 = LocalConvBlock(D, D_hidden)
    def forward(self, x): return self.c2(torch.relu(self.c1(x)))

class Hybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DirichletConv1D(N, D); self.c1 = LocalConvBlock(D, D_hidden)
    def forward(self, x): return self.c1(torch.relu(self.d1(x)))

# ──────────────────────────────────────────────
# 3. MASKED INPAINTING DATA GENERATOR (FIXED)
# ──────────────────────────────────────────────
def make_masked_signal(B, N, D=32, mask_width=5):
    """Creates spiky signals with random gaps punched out."""
    t = torch.linspace(0, 4*np.pi, N)
    base = torch.sin(1.0 * t)
    
    # Create sharp spikes
    spikes = torch.zeros(N)
    spike_idx = torch.randint(10, N-10, (N//8,)) 
    spikes[spike_idx] = torch.randn(N//8) * 2.5 
    
    clean_1d = base + spikes
    
    # ✅ FIX: Expand to [B, N, D] BEFORE masking
    clean = clean_1d.unsqueeze(0).unsqueeze(-1).expand(B, N, D).clone()
    masked = clean.clone()
    
    # Punch random holes into the BATCHED signal
    num_masks = max(1, B // 2)  
    for _ in range(num_masks):
        start = torch.randint(10, N - mask_width - 10, (1,)).item()
        # Now correctly indexes [Batch, Positions, Dim]
        masked[:, start:start+mask_width, :] = 0.0
        
    # Add noise only to non-masked regions
    noise = torch.randn_like(masked) * 0.1
    masked = masked + noise * (masked != 0).float()
    
    return clean, masked

# ──────────────────────────────────────────────
# 4. TRAINING LOOP (Masked Reconstruction)
# ──────────────────────────────────────────────
models = {
    'Pure Dirichlet': PureDirichlet().to(device),
    'Pure Conv1D': PureConv().to(device),
    'Hybrid': Hybrid().to(device)
}

print(f"\n{'='*60}\n  CAPACITY CONTROL\n{'='*60}")
for name, model in models.items():
    print(f"  {name:15} | Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"{'='*60}\n")

history = {name: [] for name in models}
optimizers = {name: torch.optim.Adam(m.parameters(), lr=2e-3) for name, m in models.items()}
loss_fn = nn.MSELoss(reduction='none') # Per-element loss to mask out gaps
STEPS = 3000

print("Training on MASKED SIGNALS (Gap-Filling)...")
start = time.time()
for step in range(STEPS):
    clean, masked = make_masked_signal(32, N, mask_width=5)
    clean, masked = clean.to(device), masked.to(device)
    
    # Only compute loss on non-zero (non-masked) positions? 
    # NO: For inpainting, we MUST compute loss ON the masked region.
    # But we can't let the model cheat by seeing the target there.
    # So we pass masked input, but calculate loss against clean target everywhere.
    
    for name, model in models.items():
        pred = model(masked)
        # Loss is computed ONLY on the masked-out regions (where input was 0)
        mask_region = (masked == 0).float()
        loss = (loss_fn(pred, clean) * mask_region).mean()
        
        optimizers[name].zero_grad(); loss.backward(); optimizers[name].step()
        history[name].append(loss.item())
        
    if (step+1) % 100 == 0:
        print(f"Step {step+1:3d} | " + " | ".join([f"{n}: {history[n][-1]:.5f}" for n in models]))
print(f"Done in {time.time()-start:.1f}s")

# ──────────────────────────────────────────────
# 5. VISUALIZATION & VERDICT
# ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for name, losses in history.items():
    axes[0,0].plot(losses, label=name, linewidth=2)
axes[0,0].set_title('Inpainting Loss (Reconstructing Gaps)', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

clean_t, masked_t = make_masked_signal(1, N, mask_width=5)
clean_t, masked_t = clean_t.to(device), masked_t.to(device)
t = np.arange(N)

axes[0,1].plot(t, masked_t[0,:,0].cpu(), alpha=0.3, color='gray', label='Masked Input')
axes[0,1].plot(t, clean_t[0,:,0].cpu(), color='black', lw=2, label='True Target')
for name, model in models.items():
    model.eval()
    with torch.no_grad(): pred = model(masked_t)
    axes[0,1].plot(t, pred[0,:,0].cpu(), lw=1.5, ls='--', label=name)
axes[0,1].set_title('Global Inpainting Result', fontweight='bold')
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

# Find a masked region to zoom in on
mask_indices = torch.where(masked_t[0,:,0] == 0)[0]
if len(mask_indices) > 0:
    center = mask_indices[len(mask_indices)//2].item()
    zs, ze = max(0, center - 10), min(N, center + 10)
else:
    zs, ze = 50, 70

axes[1,0].plot(t[zs:ze], masked_t[0,zs:ze,0].cpu(), alpha=0.3, color='gray', label='Masked')
axes[1,0].plot(t[zs:ze], clean_t[0,zs:ze,0].cpu(), color='black', lw=3, label='True Gap')
for name, model in models.items():
    with torch.no_grad(): pred = model(masked_t)
    axes[1,0].plot(t[zs:ze], pred[0,zs:ze,0].cpu(), lw=2, ls='--', label=name)
axes[1,0].set_title('ZOOM IN: Gap Reconstruction Test', fontweight='bold', color='red')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True, alpha=0.3)

final_mse = {name: history[name][-1] for name in models}
axes[1,1].bar(final_mse.keys(), final_mse.values(), color=['#9C27B0', '#4CAF50', '#FF5722'])
axes[1,1].set_title('Final Gap MSE (Lower = Better Healing)', fontweight='bold')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('experiment3_inpainting.png', dpi=150)
plt.show()

print("\n" + "="*60 + "\n  EMPIRICAL VERDICT\n" + "="*60)
best = min(final_mse, key=final_mse.get)
print(f"  Winner: {best} (Gap MSE: {final_mse[best]:.5f})")
if best == 'Hybrid':
    print("  ✅ DEFINITIVE PROOF: Hybrid successfully bridges gaps using")
    print("     local anchors + global divisor routing.")
elif best == 'Pure Dirichlet':
    print("  ⚠️  Dirichlet divisors alone can bridge gaps (surprising!)")
else:
    print("  ❌ Local convolution surprisingly handles gaps better.")
print("="*60)

In [ ]:
# ============================================================
#  EXPERIMENT 4 (ROBUST): THE TRUE SCALING KILLSHOT
#  Fixes: Memory tracking, GPU Warmup, Theoretical Baselines
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
# REPLACE LINES 14-17 WITH THIS:
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"CUDA Memory Available: {total_gb:.1f} GB\n")
# ──────────────────────────────────────────────
# 1. MODELS (Identical to previous run)
# ──────────────────────────────────────────────
class DenseAttentionBlock(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.qkv = nn.Linear(D, 3*D, bias=False)
        self.proj = nn.Linear(D, D, bias=False)
    def forward(self, X):
        B, N, D = X.shape
        q, k, v = self.qkv(X).chunk(3, dim=-1)
        attn = (q @ k.transpose(-2, -1)) / (D ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        return self.proj(attn @ v)

class HybridDirichletMixer(nn.Module):
    def __init__(self, N, D, D_hidden):
        super().__init__()
        self.N = N; self.D = D
        out_idx, in_idx, w_idx = self._build_graph(N)
        self.register_buffer('out_idx', out_idx)
        self.register_buffer('in_idx', in_idx)
        self.register_buffer('w_idx', w_idx)
        self.W_dir = nn.Parameter(torch.randn(N, D, D) * 0.02)
        self.c1 = nn.Conv1d(D, D_hidden, kernel_size=3, padding=1)
        self.c2 = nn.Conv1d(D_hidden, D, kernel_size=3, padding=1)
        
    def _build_graph(self, N):
        o, i, w = [], [], []
        for n in range(1, N + 1):
            for k in range(1, n + 1):
                if n % k == 0:
                    o.append(n-1); i.append((n//k)-1); w.append(k-1)
        return (torch.tensor(o, dtype=torch.long), 
                torch.tensor(i, dtype=torch.long), 
                torch.tensor(w, dtype=torch.long))
                
    def forward(self, X):
        B, N, D = X.shape
        num_edges = len(self.out_idx)
        chunk_size = min(num_edges, 50000)
        Y_dir = torch.zeros(B, N, D, device=X.device, dtype=X.dtype)
        
        for start in range(0, num_edges, chunk_size):
            end = min(start + chunk_size, num_edges)
            X_src = X[:, self.in_idx[start:end], :]
            W_act = self.W_dir[self.w_idx[start:end]]
            Y_chunk = torch.einsum('bed,edo->beo', X_src, W_act)
            Y_dir.index_add_(1, self.out_idx[start:end], Y_chunk)
            
        h = torch.relu(self.c1(X.transpose(1, 2)))
        Y_loc = self.c2(h).transpose(1, 2)
        return Y_dir + Y_loc

# ──────────────────────────────────────────────
# 2. ROBUST BENCHMARK LOOP
# ──────────────────────────────────────────────
D_MODEL = 64; D_HIDDEN = 128
SCALES = [128, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]

results = {'N': [], 'Dense_MB': [], 'Hybrid_MB': [], 'Dense_ms': [], 'Hybrid_ms': []}

# Theoretical baselines for visualization
theo_dense_mb = [(n**2 * D_MODEL**2 * 4) / (1024**2) for n in SCALES] # QK matrix + attn map
theo_hybrid_mb = [(n * np.log(n) * D_MODEL * 4) / (1024**2) for n in SCALES] # Edge gather tensors

print(f"{'='*70}\n  SCALING BENCHMARK | D={D_MODEL} | Batch=1\n{'='*70}")

for N in SCALES:
    torch.cuda.empty_cache(); gc.collect()
    
    # --- DENSE ATTENTION ---
    try:
        model_d = DenseAttentionBlock(D_MODEL).to(device).eval()
        x_d = torch.randn(1, N, D_MODEL, device=device)
        
        # Warmup to eliminate cold-start latency
        for _ in range(3): 
            with torch.no_grad(): model_d(x_d)
        torch.cuda.synchronize()
        
        mem_start = torch.cuda.memory_stats()['allocated_bytes.all.current']
        t0 = time.perf_counter()
        with torch.no_grad(): model_d(x_d)
        torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) * 1000
        mem_peak = (torch.cuda.memory_stats()['allocated_bytes.all.current'] - mem_start) / (1024**2)
        
        del model_d, x_d
        print(f"N={N:>6} | Dense: {mem_peak:>8.1f} MB | {dt:>8.2f} ms ✅")
        results['Dense_MB'].append(mem_peak); results['Dense_ms'].append(dt)
        
    except RuntimeError as e:
        print(f"N={N:>6} | Dense: 💥 OOM")
        results['Dense_MB'].append(np.nan); results['Dense_ms'].append(np.nan)

    # --- HYBRID DIRICHLET ---
    try:
        model_h = HybridDirichletMixer(N, D_MODEL, D_HIDDEN).to(device).eval()
        x_h = torch.randn(1, N, D_MODEL, device=device)
        
        for _ in range(3): 
            with torch.no_grad(): model_h(x_h)
        torch.cuda.synchronize()
        
        mem_start = torch.cuda.memory_stats()['allocated_bytes.all.current']
        t0 = time.perf_counter()
        with torch.no_grad(): model_h(x_h)
        torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) * 1000
        mem_peak = (torch.cuda.memory_stats()['allocated_bytes.all.current'] - mem_start) / (1024**2)
        
        del model_h, x_h
        print(f"N={N:>6} | Hybrid: {mem_peak:>8.1f} MB | {dt:>8.2f} ms ✅")
        results['Hybrid_MB'].append(mem_peak); results['Hybrid_ms'].append(dt)
        
    except RuntimeError as e:
        print(f"N={N:>6} | Hybrid: 💥 OOM")
        results['Hybrid_MB'].append(np.nan); results['Hybrid_ms'].append(np.nan)

    results['N'].append(N)

# ──────────────────────────────────────────────
# 3. VISUALIZATION WITH THEORETICAL BASELINES
# ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# VRAM Plot
axes[0].plot(results['N'], theo_dense_mb, ':', color='#FF5722', alpha=0.5, label='Theoretical Dense $O(N^2)$')
# REPLACE LINE 148 WITH THIS (raw string for LaTeX):
axes[0].plot(results['N'], theo_hybrid_mb, ':', color='#4CAF50', alpha=0.5, label=r'Theoretical Hybrid $O(N \log N)$')
axes[0].plot(results['N'], results['Dense_MB'], 'o-', color='#FF5722', label='Dense Actual', linewidth=2)
axes[0].plot(results['N'], results['Hybrid_MB'], 's--', color='#4CAF50', label='Hybrid Actual', linewidth=2)
axes[0].set_title('VRAM Usage vs Sequence Length', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Sequence Length (N)'); axes[0].set_ylabel('Memory (MB)')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

# Latency Plot
axes[1].plot(results['N'], results['Dense_ms'], 'o-', color='#FF5722', label='Dense Attention', linewidth=2)
axes[1].plot(results['N'], results['Hybrid_ms'], 's--', color='#4CAF50', label='Hybrid Dirichlet', linewidth=2)
axes[1].set_title('Inference Latency vs Sequence Length', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Sequence Length (N)'); axes[1].set_ylabel('Time (ms)')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiment4_robust_scaling.png', dpi=150)
plt.show()

# VERDICT
last_d = next((i for i,m in enumerate(results['Dense_MB']) if not np.isnan(m)), None)
last_h = next((i for i,m in enumerate(results['Hybrid_MB']) if not np.isnan(m)), None)
print(f"\n{'='*70}\n  SCALING VERDICT\n{'='*70}")
if last_h > last_d:
    print(f"  ✅ KILLSHOT CONFIRMED")
    print(f"     Dense died at N={results['N'][last_d]}")
    print(f"     Hybrid survived to N={results['N'][last_h]}")
else:
    print("  ⚠️  Check memory tracking or increase D_MODEL")
print("="*70)

In [4]:
# ============================================================
#  EXPERIMENT 5: FIRST LIGHT - CAUSAL DIRICHLET-GPT (FINAL)
#  Language Modeling via Prime Signatures (No PosEmb, No Attn)
# ============================================================
import os; os.environ["TORCHDYNAMO_DISABLE"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import urllib.request
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ──────────────────────────────────────────────
# 1. DATA LOADING (TinyShakespeare)
# ──────────────────────────────────────────────
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    with urllib.request.urlopen(url) as f:
        text = f.read().decode('utf-8')[:30000]
except:
    text = "The quick brown fox jumps over the lazy dog. " * 500

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long).to(device)
print(f"Vocab: {vocab_size}, Dataset: {len(data)} chars")

# ──────────────────────────────────────────────
# 2. THE PRIME SIGNATURE EMBEDDER
# ──────────────────────────────────────────────
PRIMES = [2, 3, 5, 7, 11, 13, 17, 19]
MAX_CTX = 256

def get_prime_signature(n):
    sig = []
    temp = n
    for p in PRIMES:
        count = 0
        while temp % p == 0:
            count += 1
            temp //= p
        sig.append(count)
    sig.append(np.log(temp + 1e-9))
    return torch.tensor(sig, dtype=torch.float32)

prime_sigs = torch.stack([get_prime_signature(i+1) for i in range(MAX_CTX)]).to(device)
SIG_DIM = len(PRIMES) + 1

# ──────────────────────────────────────────────
# 3. HYPERNETWORK-COMPRESSED DIRICHLET MIXER
# ──────────────────────────────────────────────
class HyperDirichletLayer(nn.Module):
    def __init__(self, D, sig_dim=SIG_DIM):
        super().__init__()
        self.D = D
        self.hypernet = nn.Sequential(
            nn.Linear(sig_dim, 64),
            nn.GELU(),
            nn.Linear(64, D*D)
        )
        
    def forward(self, X, pos_sigs):
        B, N, D = X.shape
        
        # Handle batched vs unbatched signatures
        if pos_sigs.dim() == 3:
            sig_flat = pos_sigs.reshape(-1, pos_sigs.shape[-1])
            W_flat = self.hypernet(sig_flat)
            W = W_flat.view(B, N, D, D)
        else:
            W_flat = self.hypernet(pos_sigs[:N])
            W = W_flat.view(N, D, D).unsqueeze(0).expand(B, -1, -1, -1)
        
        # Build divisor graph dynamically
        out_idx, in_idx, w_idx = [], [], []
        for n in range(1, N + 1):
            for k in range(1, n + 1):
                if n % k == 0:
                    out_idx.append(n-1); in_idx.append((n//k)-1); w_idx.append(k-1)
                    
        o_i = torch.tensor(out_idx, dtype=torch.long, device=X.device)
        i_i = torch.tensor(in_idx, dtype=torch.long, device=X.device)
        w_i = torch.tensor(w_idx, dtype=torch.long, device=X.device)
        
        X_src = X[:, i_i, :]
        W_act = W[:, w_i, :, :]
        Y_edges = torch.einsum('bed,bedo->beo', X_src, W_act)
        
        Y = torch.zeros(B, N, D, device=X.device, dtype=X.dtype)
        Y.index_add_(1, o_i, Y_edges)
        return Y

class CausalLocalPatch(nn.Module):
    def __init__(self, D, D_hidden):
        super().__init__()
        self.c1 = nn.Conv1d(D, D_hidden, kernel_size=3, padding=0)
        self.c2 = nn.Conv1d(D_hidden, D, kernel_size=3, padding=0)
        
    def forward(self, X):
        x_t = X.transpose(1, 2)
        h = F.gelu(self.c1(F.pad(x_t, (2, 0))))
        y = self.c2(F.pad(h, (2, 0)))
        y = y[:, :, :X.shape[1]]
        return y.transpose(1, 2)

# ──────────────────────────────────────────────
# 4. THE DIRICHLET-GPT ARCHITECTURE
# ──────────────────────────────────────────────
BLOCK_SIZE = 128
D_MODEL = 64
D_HIDDEN = 128

class DirichletGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, D_MODEL)
        self.dirichlet = HyperDirichletLayer(D_MODEL)
        self.local_patch = CausalLocalPatch(D_MODEL, D_HIDDEN)
        self.ln = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, vocab_size)
        
    def forward(self, idx):
        B, T = idx.shape
        x = self.token_emb(idx)
        
        if T < BLOCK_SIZE:
            pad = torch.zeros(B, BLOCK_SIZE-T, D_MODEL, device=device)
            x = torch.cat([x, pad], dim=1)
            
        pos_sigs = prime_sigs[:T].unsqueeze(0).expand(B, -1, -1)
        
        d_out = self.dirichlet(x, pos_sigs)
        l_out = self.local_patch(x)
        
        x = self.ln(d_out + l_out)
        logits = self.head(x[:, :T, :])
        return logits

# ──────────────────────────────────────────────
# 5. TRAINING LOOP
# ──────────────────────────────────────────────
model = DirichletGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
STEPS = 20000

def get_batch():
    ix = torch.randint(len(data) - BLOCK_SIZE, (16,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x, y

print(f"\n{'='*60}\n  TRAINING DIRICHLET-GPT\n{'='*60}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"No Positional Embeddings | No Attention Matrices\n")

start = time.time()
for step in range(STEPS):
    x, y = get_batch()
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    if (step+1) % 200 == 0:
        print(f"Step {step+1:4d} | Loss: {loss.item():.4f} | Time: {time.time()-start:.1f}s")

print(f"\nTraining complete in {time.time()-start:.1f}s")

# ──────────────────────────────────────────────
# SAVE CHECKPOINT AFTER TRAINING
# ──────────────────────────────────────────────
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'final_loss': loss.item(),
    'step': STEPS
}
torch.save(checkpoint, 'dirichlet_gpt_first_light.pt')
print(f"\n💾 Checkpoint saved! Size: {os.path.getsize('dirichlet_gpt_first_light.pt') / 1024:.1f} KB")

# ──────────────────────────────────────────────
# 6. GENERATION: FIRST LIGHT
# ──────────────────────────────────────────────
@torch.no_grad()
def generate(start_str, max_new_tokens=200):
    model.eval()
    idx = torch.tensor([[stoi[c] for c in start_str]], dtype=torch.long, device=device)
    abs_pos = len(start_str) 
    
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        T = idx_cond.shape[1]
        
        # Compute absolute position range for current window
        start_abs = max(0, abs_pos - T + 1)
        end_abs = min(MAX_CTX, abs_pos + 1)
        
        # Ensure we always have valid signatures
        if end_abs <= start_abs:
            start_abs = max(0, MAX_CTX - T)
            end_abs = MAX_CTX
            
        # Get signatures for THIS specific window's absolute positions
        pos_sigs = prime_sigs[start_abs:end_abs].to(device)
        
        # Pass directly to model - let forward handle batching
        logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
        abs_pos += 1
        
    return ''.join([itos[i.item()] for i in idx[0]])

print("\n" + "="*60)
print("  FIRST LIGHT: GENERATED TEXT")
print("="*60)

prompts = ["The ", "To be ", "King "]
for prompt in prompts:
    try:
        generated = generate(prompt, 150)
        print(f"\n[Prompt: '{prompt}']\n{generated}")
    except Exception as e:
        print(f"\n[Prompt: '{prompt}'] → Error: {str(e)[:80]}")
    print("-"*60)

print("\n✅ FIRST LIGHT ACHIEVED: Primes are speaking English.")
print("="*60)

Device: cuda
Vocab: 58, Dataset: 30000 chars


AttributeError: module 'torch' has no attribute '_utils'

In [5]:
# FINAL DUAL-T4 DIRICHLET-GPT (12M PARAMS)
# Targets GPU 1 on Kaggle Dual-T4 Setup | Fixed Top-P + Prime-Aware Temp
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["PYTORCH_JIT"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import json
from datasets import load_dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

# ✅ TARGET GPU 1 ON KAGGLE DUAL-T4 SETUP
device = torch.device('cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')
print(f"Target Device: {device} | Total GPUs: {torch.cuda.device_count()}")
print(f"VRAM Available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.1f} GB")
# ──────────────────────────────────────────────
# 1. BULLETPROOF DATA & TOKENIZER
# ──────────────────────────────────────────────
print("1. Loading TinyStories...")
ds = load_dataset("roneneldan/TinyStories", split="train[:50000]")
texts = [item["text"] for item in ds]
full_text = "\n".join(texts)

print("2. Training BPE on FULL text...")
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.Whitespace()
tok.train_from_iterator([full_text], trainers.BpeTrainer(vocab_size=512, special_tokens=["<|endoftext|>"]))
tok.add_special_tokens(["<|endoftext|>"])

V = tok.get_vocab_size()
EOS = tok.token_to_id("<|endoftext|>")
print(f"   Vocab: {V}, EOS ID: {EOS}")

print("3. Encoding safely on CPU...")
all_t = []
for t in texts:
    ids = tok.encode(t).ids + [EOS]
    valid_ids = [i for i in ids if 0 <= i < V]
    all_t.extend(valid_ids)

min_id, max_id = min(all_t), max(all_t)
print(f"   CPU Validation: Min={min_id}, Max={max_id}")
if min_id < 0 or max_id >= V:
    raise ValueError(f"INVALID TOKENS! Min: {min_id}, Max: {max_id}, Vocab: {V}")

data_cpu = torch.tensor(all_t, dtype=torch.long)
data = data_cpu.to(device)
print(f"   ✅ Data on GPU. Shape: {data.shape}")

def enc(t): return [i for i in tok.encode(t).ids + [EOS] if 0 <= i < V]
def dec(ids): return tok.decode([i for i in ids if 0 <= i < V])

# ──────────────────────────────────────────────
# 2. PRIME SIGNATURES & GRAPH
# ──────────────────────────────────────────────
PRIMES = [2, 3, 5, 7, 11, 13, 17, 19]
SIG_D = len(PRIMES) + 1
CTX = 128

def psig(n):
    s, t = [], n
    for p in PRIMES:
        c = 0
        while t % p == 0: c += 1; t //= p
        s.append(c)
    s.append(np.log(t + 1e-9))
    return torch.tensor(s, dtype=torch.float32)

psigs_pre = torch.stack([psig(i+1) for i in range(CTX)]).to(device)
def psig_otf(n): return psig(n).to(device)

div_o, div_i, div_w = [], [], []
for n in range(1, CTX+1):
    for k in range(1, n+1):
        if n % k == 0:
            div_o.append(n-1); div_i.append((n//k)-1); div_w.append(k-1)

DO = torch.tensor(div_o, dtype=torch.long, device=device)
DI = torch.tensor(div_i, dtype=torch.long, device=device)
DW = torch.tensor(div_w, dtype=torch.long, device=device)
print(f"4. Precomputed Edges: {len(div_o)} | SIG_D: {SIG_D}")

# ──────────────────────────────────────────────
# 3. FULL 12M PARAMETER MODEL
# ──────────────────────────────────────────────
D_MODEL = 256      # Restored for Dual-T4
D_HIDDEN = 512
N_LAYERS = 6
D_SQ = D_MODEL * D_MODEL

class HyperDir(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(SIG_D, 128),
            nn.GELU(),
            nn.Linear(128, D_SQ)
        )
        self.register_buffer('oi', DO.clone())
        self.register_buffer('ii', DI.clone())
        self.register_buffer('wi', DW.clone())
        
    def forward(self, X, ps):
        B, N, D = X.shape
        mask = self.oi < N
        oi = self.oi[mask]; ii = self.ii[mask]; wi = self.wi[mask]
        
        # 1. Feed ONLY 2D absolute signatures. No Batch copying!
        # [N, SIG_D] -> [N, D_SQ] -> [N, D, D]
        wf = self.net(ps[:N]).view(N, D, D)  
        
        # 2. Extract active edges safely in pure 3D! 
        # Size becomes just [645, 256, 256] -> only 169 Megabytes total!
        W_active = wf[wi]                    
            
        # 3. Einstein Summation natively broadcasts over Batch 'b' cleanly.
        # X is [b, e, d] and W_active is [e, d, o] -> output [b, e, o]
        Ye = torch.einsum('bed,edo->beo', X[:, ii, :], W_active)
        
        Y = torch.zeros(B, N, D, device=X.device, dtype=X.dtype)
        Y.index_add_(1, oi, Ye)
        return Y

class LocalPatch(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(D_MODEL, D_HIDDEN, 3, padding=0)
        self.c2 = nn.Conv1d(D_HIDDEN, D_MODEL, 3, padding=0)
    def forward(self, X):
        xt = X.transpose(1,2)
        h = F.gelu(self.c1(F.pad(xt, (2,0))))
        y = self.c2(F.pad(h, (2,0)))[:, :, :X.shape[1]]
        return y.transpose(1,2)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.d = HyperDir()
        self.l = LocalPatch()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.ln2 = nn.LayerNorm(D_MODEL)
    def forward(self, X, ps):
        X = X + self.d(self.ln1(X), ps)
        X = X + self.l(self.ln2(X))
        return X

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D_MODEL)
        self.blks = nn.ModuleList([Block() for _ in range(N_LAYERS)])
        self.lnf = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, V)
    def forward(self, idx, ps=None):
        B, T = idx.shape
        x = self.emb(idx)
        if ps is None:
            ps = psigs_pre[:T]  # KEEP 2D! Zero-copy expand happens in HyperDir else-branch
        for b in self.blks: x = b(x, ps)
        return self.head(self.lnf(x))

model = GPT().to(device)
params = sum(p.numel() for p in model.parameters())
print(f"\n{'='*60}\n  DUAL-T4 DIRICHLET-GPT (12M)\n{'='*60}")
print(f"Params: {params:,} | d_model={D_MODEL} | layers={N_LAYERS}")

# ──────────────────────────────────────────────
# 4. TRAINING (BATCH SIZE 64 FOR DUAL-T4)
# ──────────────────────────────────────────────
LR, B1, B2, EPS, WD = 2e-4, 0.9, 0.999, 1e-8, 0.01
m_b = {n: torch.zeros_like(p) for n,p in model.named_parameters()}
v_b = {n: torch.zeros_like(p) for n,p in model.named_parameters()}

def batch():
    ix = torch.randint(len(data)-CTX, (64,)) # Increased for Dual-T4
    x = torch.stack([data[i:i+CTX] for i in ix])
    y = torch.stack([data[i+1:i+CTX+1] for i in ix])
    return x, y

STEPS = 15000
start = time.time(); best = float('inf')
SD = "ar_dualt4_ckpt"; os.makedirs(SD, exist_ok=True)

print("\nStarting Training...")
for step in range(1, STEPS+1):
    x, y = batch()
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    
    model.zero_grad(); loss.backward()
    
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.grad is None: continue
            g = p.grad.data
            p.data.mul_(1 - LR*WD)
            m_b[n].mul_(B1).add_(g, alpha=1-B1)
            v_b[n].mul_(B2).addcmul_(g, g, value=1-B2)
            mh = m_b[n] / (1 - B1**step)
            vh = v_b[n] / (1 - B2**step)
            p.data.addcdiv_(mh, vh.sqrt().add(EPS), value=-LR)
    
    if step % 500 == 0:
        el = time.time()-start
        print(f"Step {step:5d} | Loss: {loss.item():.4f} | {el:.1f}s")
        if loss.item() < best:
            best = loss.item()
            meta = {"loss": best, "step": step, "p": {}}
            for n, p in model.named_parameters():
                fn = f"{SD}/{n.replace('.','_')}.npy"
                np.save(fn, p.detach().cpu().numpy())
                meta["p"][n] = {"f": fn, "s": list(p.shape), "d": str(p.dtype)}
            with open(f"{SD}/meta.json","w") as f: json.dump(meta, f)
            print(f"  💾 Saved ({best:.4f})")

print(f"\nDone: {time.time()-start:.1f}s | Best Loss: {best:.4f}")

# ──────────────────────────────────────────────
# 5. FIXED GENERATION
# ──────────────────────────────────────────────
@torch.no_grad()
def gen_refined(prompt, mx=200, top_p=0.9):
    model.eval()
    ts = enc(prompt)
    idx = torch.tensor([ts], dtype=torch.long, device=device)
    
    for step in range(mx):
        c = idx[:, -CTX:]
        T = c.shape[1]
        abs_pos = len(ts)
        
        if abs_pos < CTX:
            sig = psigs_pre[abs_pos]
        else:
            sig = psig_otf(abs_pos + 1)
        
        complexity = sig[:-1].sum().item()
        temp = 0.7 + 0.8 / (1.0 + complexity)
        
        lo = model(c)[:, -1, :] / temp
        probs = F.softmax(lo, dim=-1)
        
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(sorted_probs, dim=-1)
        
        mask = cumsum > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = 0
        
        sorted_probs[mask] = 0.0
        probs_clean = torch.zeros_like(probs).scatter_(-1, sorted_idx, sorted_probs)
        probs_clean /= probs_clean.sum() + 1e-10
        
        nx = torch.multinomial(probs_clean, 1)
        idx = torch.cat((idx, nx), dim=1)
        ts.append(nx.item())
        if nx.item() == EOS: break
        
    return dec(ts)

print("\n" + "="*60 + "\n  DUAL-T4 GENERATION TEST\n" + "="*60)
prompts = [
    "Once upon a time, there was a ",
    "The little girl wanted to ",
    "A big red dragon lived in a ",
    "Tommy went to the "
]

for p in prompts:
    try:
        out = gen_refined(p, mx=150)
        print(f"\n[Prompt]: {p}")
        print(f"[Generated]: {out}")
    except Exception as e:
        print(f"[{p}] → Error: {str(e)[:80]}")
    print("-"*60)

print("\n✅ COMPLETE. 12M PRIME ARITHMETIC IS SPEAKING.")

Target Device: cuda:1 | Total GPUs: 2
VRAM Available: 15.6 GB
1. Loading TinyStories...
2. Training BPE on FULL text...



   Vocab: 512, EOS ID: 0
3. Encoding safely on CPU...
   CPU Validation: Min=0, Max=511
   ✅ Data on GPU. Shape: torch.Size([15444411])
4. Precomputed Edges: 645 | SIG_D: 9

  DUAL-T4 DIRICHLET-GPT (12M)
Params: 55,725,056 | d_model=256 | layers=6

Starting Training...
Step   500 | Loss: 3.8808 | 101.1s
  💾 Saved (3.8808)
Step  1000 | Loss: 3.0837 | 202.7s
  💾 Saved (3.0837)
Step  1500 | Loss: 2.6624 | 304.3s
  💾 Saved (2.6624)
Step  2000 | Loss: 2.5016 | 406.0s
  💾 Saved (2.5016)
Step  2500 | Loss: 2.3979 | 507.6s
  💾 Saved (2.3979)
Step  3000 | Loss: 2.3310 | 609.3s
  💾 Saved (2.3310)
Step  3500 | Loss: 2.2342 | 710.9s
  💾 Saved (2.2342)
Step  4000 | Loss: 2.2153 | 812.6s
  💾 Saved (2.2153)
Step  4500 | Loss: 2.1252 | 914.2s
  💾 Saved (2.1252)
Step  5000 | Loss: 2.0796 | 1015.9s
  💾 Saved (2.0796)
Step  5500 | Loss: 2.0830 | 1117.5s
Step  6000 | Loss: 2.0180 | 12